In [16]:
# ============================================
# DB 연결 및 설정
# ============================================

import sqlite3
import pandas as pd

# DB 파일 연결 (없으면 생성됨)
conn = sqlite3.connect("restaurant.db")
cursor = conn.cursor()

# SQLite는 기본적으로 FOREIGN KEY가 꺼져있음 → 반드시 활성화
conn.execute("PRAGMA foreign_keys = ON;")

print("✅ DB 연결 완료")

✅ DB 연결 완료


In [17]:
# 테이블 목록
tables = [
    "restaurant", "users", "food", "category", "tag",
    "menu", "review",
    "rel_restaurant_category", "rel_restaurant_tag", "rel_review_tag"
]

In [18]:
# ============================================
# 기존 테이블 삭제 (초기화용)
# 필요할 때만 실행
# ============================================

for t in reversed(tables):
    cursor.executescript(f"DROP TABLE IF EXISTS {t}")

conn.commit()

print("🧹 기존 테이블 삭제 완료")

🧹 기존 테이블 삭제 완료


In [19]:
# ============================================
# USER 테이블 생성
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    user_code TEXT PRIMARY KEY,
    name TEXT,
    avg_score REAL,
    review_cnt INTEGER,
    follower_cnt INTEGER
);
""")

conn.commit()
print("✅ USERS 테이블 생성 완료")

✅ USERS 테이블 생성 완료


In [20]:
# ============================================
# RESTAURANT 테이블 생성
# 위치 기반 검색을 위해 lat/lng 포함
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS restaurant (
    restaurant_code TEXT PRIMARY KEY,
    name TEXT,
    img_link TEXT,
    region TEXT,
    address TEXT,
    lat REAL,
    lng REAL,
    open_time TEXT,
    close_time TEXT,
    tel_no TEXT
);
""")

conn.commit()
print("✅ RESTAURANT 테이블 생성 완료")

✅ RESTAURANT 테이블 생성 완료


In [21]:
# ============================================
# FOOD 테이블 (선택 유지)
# 음식 카테고리/유사도 분석용
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS food (
    food_code TEXT PRIMARY KEY,
    name TEXT,
    description TEXT,
    embedding TEXT
);
""")

conn.commit()
print("✅ FOOD 테이블 생성 완료")

✅ FOOD 테이블 생성 완료


In [22]:
# ============================================
# MENU 테이블 생성
# food_code 제거 → 메뉴는 식당 종속 구조
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS menu (
    menu_code TEXT PRIMARY KEY,
    restaurant_code TEXT,
    food_code TEXT,
    name TEXT,
    price INTEGER,
    description TEXT,
    prompted_description TEXT,
    embedding TEXT,
    FOREIGN KEY (restaurant_code) REFERENCES restaurant(restaurant_code),
    FOREIGN KEY (food_code) REFERENCES food(food_code)
);
""")

conn.commit()
print("✅ MENU 테이블 생성 완료")

✅ MENU 테이블 생성 완료


In [23]:
# ============================================
# CATEGORY 테이블
# 고정 분류 (한식, 중식 등)
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS category (
    category_code TEXT PRIMARY KEY,
    name TEXT,
    description TEXT,
    embedding TEXT
);
""")

conn.commit()
print("✅ CATEGORY 테이블 생성 완료")

✅ CATEGORY 테이블 생성 완료


In [24]:
# ============================================
# TAG 테이블
# 유동적 분류 (데이트, 혼밥, 가성비 등)
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS tag (
    tag_code TEXT PRIMARY KEY,
    name TEXT,
    description TEXT,
    embedding TEXT
);
""")

conn.commit()
print("✅ TAG 테이블 생성 완료")

✅ TAG 테이블 생성 완료


In [25]:
# ============================================
# REVIEW 테이블
# 평점 + 세부 평가 + 텍스트 + embedding 포함
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS review (
    review_code TEXT PRIMARY KEY,
    restaurant_code TEXT,
    user_code TEXT,
    score REAL,
    taste_level INTEGER,
    price_level INTEGER,
    service_level INTEGER,
    content TEXT,
    menu TEXT,
    embedding TEXT,
    FOREIGN KEY (restaurant_code) REFERENCES restaurant(restaurant_code),
    FOREIGN KEY (user_code) REFERENCES users(user_code)
);
""")

conn.commit()
print("✅ REVIEW 테이블 생성 완료")

✅ REVIEW 테이블 생성 완료


In [26]:
# ============================================
# RESTAURANT - CATEGORY (다대다 관계)
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS rel_restaurant_category (
    restaurant_code TEXT,
    category_code TEXT,
    PRIMARY KEY (restaurant_code, category_code),
    FOREIGN KEY (restaurant_code) REFERENCES restaurant(restaurant_code),
    FOREIGN KEY (category_code) REFERENCES category(category_code)
);
""")

conn.commit()
print("✅ RESTAURANT-CATEGORY 관계 생성 완료")

✅ RESTAURANT-CATEGORY 관계 생성 완료


In [27]:
# ============================================
# RESTAURANT - TAG (다대다 관계)
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS rel_restaurant_tag (
    restaurant_code TEXT,
    tag_code TEXT,
    PRIMARY KEY (restaurant_code, tag_code),
    FOREIGN KEY (restaurant_code) REFERENCES restaurant(restaurant_code),
    FOREIGN KEY (tag_code) REFERENCES tag(tag_code)
);
""")

conn.commit()
print("✅ RESTAURANT-TAG 관계 생성 완료")

✅ RESTAURANT-TAG 관계 생성 완료


In [28]:
# ============================================
# REVIEW - TAG (다대다 관계)
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS rel_review_tag (
    review_code TEXT,
    tag_code TEXT,
    PRIMARY KEY (review_code, tag_code),
    FOREIGN KEY (review_code) REFERENCES review(review_code),
    FOREIGN KEY (tag_code) REFERENCES tag(tag_code)
);
""")

conn.commit()
print("✅ REVIEW-TAG 관계 생성 완료")

✅ REVIEW-TAG 관계 생성 완료


In [29]:
# ============================================
# 생성된 테이블 확인
# ============================================

tables_df = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables_df

,name
0,users
1,restaurant
2,food
3,menu
4,category
5,tag
6,review
7,rel_restaurant_category
8,rel_restaurant_tag
9,rel_review_tag


In [30]:
# ============================================
# CSV → SQLite 적재
# ============================================
import pandas as pd

filenames = [
    "restaurant", "users", "emb_food", "emb_category", "emb_tag",
    "emb_menu", "emb_review",
    "rel_res_cat", "rel_res_tag", "rel_rev_tag"
]

# CSV → DB 적재
for f, t in zip(filenames, tables):
    path = "db_csv/" + f + ".csv"
    df = pd.read_csv(path)

    df.to_sql(t, conn, if_exists="append", index=False)
    print(f"✅ {t} INSERT 완료 ({len(df)} rows)")

# 커밋
conn.commit()
print("🎉 적재 완료")

✅ restaurant INSERT 완료 (100 rows)
✅ users INSERT 완료 (171 rows)
✅ food INSERT 완료 (323 rows)
✅ category INSERT 완료 (123 rows)
✅ tag INSERT 완료 (143 rows)
✅ menu INSERT 완료 (2008 rows)
✅ review INSERT 완료 (422 rows)
✅ rel_restaurant_category INSERT 완료 (176 rows)
✅ rel_restaurant_tag INSERT 완료 (625 rows)
✅ rel_review_tag INSERT 완료 (2336 rows)
🎉 적재 완료


In [31]:
import sqlite3

conn = sqlite3.connect("restaurant.db")
conn.execute("PRAGMA foreign_keys = ON;")

cursor = conn.cursor()

cursor.execute("SELECT * FROM restaurant")
rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

('RES0000', '유태우스시', 'https://d12zq4w4guyljn.cloudfront.net/300_300_20251221075826166_photo_c2f5e0670740.webp', '신대방삼거리역', '서울특별시 동작구 보라매로 113 1층 유태우스시', 37.4994633095093, 126.927896614044, '11:00', '22:00', '0507-1358-1246')
('RES0001', '듀윗', 'https://d12zq4w4guyljn.cloudfront.net/300_300_20250408030027357_photo_7a77fcde6cd2.webp', '신대방삼거리역', '서울특별시 동작구 대방동1길 10 1층', 37.5003637668422, 126.92671225982, '08:00', '21:00', '0507-2093-6672')
('RES0002', '한제소곱창전골 신대방삼거리역점', 'https://d12zq4w4guyljn.cloudfront.net/300_300_20260303032143_menu1_3c232e2fcc86.webp', '신대방삼거리역', '서울특별시 동작구 여의대방로22길 121 상가동 1층 1444호, 1445호', 37.499106, 126.926924, '16:00', '00:00', '0507-1359-9161')
('RES0003', '프로스퍼', 'https://d12zq4w4guyljn.cloudfront.net/300_300_20250921111001_photo1_394db3cb2fa0.webp', '신대방삼거리역', '서울특별시 동작구 여의대방로24길 101 1층', 37.4990877047725, 126.925611349157, '10:00', '00:00', '0507-1361-3276')
('RES0004', '일진아구찜', 'https://d12zq4w4guyljn.cloudfront.net/300_300_20250206042202346_photo_75637b91f